# Improved LightGBM Training on EMBER Dataset
This notebook implements an advanced training pipeline to yield higher Accuracy and AUC scores compared to the base model.
**Key Improvements:**
1. **Tuned Hyperparameters:** Lower `learning_rate` (0.01) coupled with larger `num_leaves` (2048) allows the model to learn more complex patterns without missing fine details.
2. **More Boost Rounds:** Increased from 50 to 100 trees per chunk.
3. **Epoch Training:** The model makes multiple passes (`EPOCHS`) through the entire dataset to maximize learning.

In [1]:
import os
import sys
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# Add local venv site-packages to path so thrember can be imported
base_dir = os.getcwd()
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages):
        sys.path.append(site_packages)
        break

try:
    from thrember.features import PEFeatureExtractor
    print("Features extractor imported successfully.")
except ImportError:
    print("Warning: 'thrember' not found. Will use default feature dimension (2381).")
    class PEFeatureExtractor:
        dim = 2381

Features extractor imported successfully.


In [2]:
# --- IMPROVED CONFIGURATION ---
DATASET_DIR = r"Z:\ember2024_train_data"
CHUNK_SIZE = 10000                  # Strict 10k samples per chunk to avoid Notebook kernel crash
EPOCHS = 2                          # Number of full passes over the dataset
BOOST_ROUNDS_PER_CHUNK = 5          # 5 trees per chunk to keep the model strictly within memory limits

# Tuned LightGBM parameters for higher accuracy
params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,          # Faster learning to compensate for fewer trees per chunk
    "num_leaves": 1024,             # Safe tree size
    "max_depth": 15,                # Cap depth to prevent overfitting with large num_leaves
    "feature_fraction": 0.7,        # Use 70% of features per tree (adds robustness)
    "bagging_fraction": 0.8,        # Use 80% of data per tree
    "bagging_freq": 5,              # Perform bagging every 5 iterations
    "verbose": -1,
    "n_jobs": -1
}

categorical_features = [2, 3, 4, 5, 6, 701, 702]

In [3]:
print(f"Preparing data from {DATASET_DIR}...")

X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

if not os.path.exists(X_path) or not os.path.exists(y_path):
    raise FileNotFoundError("Error: Training data not found!")

extractor = PEFeatureExtractor()
ndim = extractor.dim

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)

# Open memory-mapped files
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

train_nrows = int(nrows * 0.9)
print(f"Total rows: {nrows} | Training on {train_nrows} samples | Validation set aside.")

Preparing data from Z:\ember2024_train_data...
Total rows: 5252000 | Training on 4726800 samples | Validation set aside.


### Multi-Epoch Training Loop
This loop runs through the dataset multiple times. It tracks both the current epoch and chunk in the checkpoint state file so it can recover perfectly if interrupted.

In [4]:
model = None
start_epoch = 0
start_chunk_idx = 0

ckpt_path = os.path.join(DATASET_DIR, "ember_model_tuned_checkpoint.txt")
state_path = os.path.join(DATASET_DIR, "ember_tuned_state.txt")

# --- SMART RESUME (Epoch & Chunk Aware) ---
if os.path.exists(ckpt_path) and os.path.exists(state_path):
    print(f"Found existing tuned checkpoint!")
    try:
        model = lgb.Booster(model_file=ckpt_path)
        with open(state_path, "r") as f:
            state_data = f.read().split(",")
            start_epoch = int(state_data[0])
            start_chunk_idx = int(state_data[1])
        print(f"Resuming at Epoch {start_epoch + 1}, Chunk {start_chunk_idx}")
    except Exception as e:
        print(f"Error loading checkpoint: {e}. Starting fresh.")
        model, start_epoch, start_chunk_idx = None, 0, 0
else:
    print("Starting fresh tuned training.")

total_chunks = (train_nrows + CHUNK_SIZE - 1) // CHUNK_SIZE

# --- THE TRAINING LOOP ---
for epoch in range(start_epoch, EPOCHS):
    print("\n" + "="*45)
    print(f"🚀 STARTING EPOCH {epoch + 1} / {EPOCHS}")
    print("="*45)
    
    current_chunk = 0
    
    for start_idx in range(0, train_nrows, CHUNK_SIZE):
        # Skip previously trained chunks if resuming mid-epoch
        if epoch == start_epoch and current_chunk < start_chunk_idx:
            current_chunk += 1
            continue

        end_idx = min(start_idx + CHUNK_SIZE, train_nrows)
        print(f"[E{epoch+1}] Chunk {current_chunk}/{total_chunks}: Rows {start_idx} to {end_idx}...")
        
        # Load and filter
        X_chunk = np.array(X_memmap[start_idx:end_idx])
        y_chunk = np.array(y_memmap[start_idx:end_idx])
        
        valid_idx = y_chunk != -1
        X_chunk = X_chunk[valid_idx]
        y_chunk = y_chunk[valid_idx]
        
        if len(y_chunk) == 0:
            current_chunk += 1
            continue
            
        train_data = lgb.Dataset(X_chunk, label=y_chunk, categorical_feature=categorical_features, free_raw_data=False)
        
        # Incrementally update the model
        model = lgb.train(
            params,
            train_data,
            num_boost_round=BOOST_ROUNDS_PER_CHUNK, 
            init_model=model,  
            keep_training_booster=True
        )
        
        current_chunk += 1
        
        # Save state & model every 5 chunks
        if current_chunk % 5 == 0:
            model.save_model(ckpt_path)
            with open(state_path, "w") as f:
                f.write(f"{epoch},{current_chunk}")
            print(f"  --> Checkpoint Saved.")

    # Reset chunk index for the next epoch iteration
    start_chunk_idx = 0 

print("\n🎉 TUNED TRAINING COMPLETE!")

Found existing tuned checkpoint!
Resuming at Epoch 2, Chunk 335

🚀 STARTING EPOCH 2 / 2
[E2] Chunk 335/473: Rows 3350000 to 3360000...
[E2] Chunk 336/473: Rows 3360000 to 3370000...
[E2] Chunk 337/473: Rows 3370000 to 3380000...
[E2] Chunk 338/473: Rows 3380000 to 3390000...
[E2] Chunk 339/473: Rows 3390000 to 3400000...
  --> Checkpoint Saved.
[E2] Chunk 340/473: Rows 3400000 to 3410000...
[E2] Chunk 341/473: Rows 3410000 to 3420000...
[E2] Chunk 342/473: Rows 3420000 to 3430000...
[E2] Chunk 343/473: Rows 3430000 to 3440000...
[E2] Chunk 344/473: Rows 3440000 to 3450000...
  --> Checkpoint Saved.
[E2] Chunk 345/473: Rows 3450000 to 3460000...
[E2] Chunk 346/473: Rows 3460000 to 3470000...
[E2] Chunk 347/473: Rows 3470000 to 3480000...
[E2] Chunk 348/473: Rows 3480000 to 3490000...
[E2] Chunk 349/473: Rows 3490000 to 3500000...
  --> Checkpoint Saved.
[E2] Chunk 350/473: Rows 3500000 to 3510000...
[E2] Chunk 351/473: Rows 3510000 to 3520000...
[E2] Chunk 352/473: Rows 3520000 to 35300

In [5]:
# Save the final tuned model specifically
final_save_path = os.path.join(DATASET_DIR, "ember_model_tuned_full.txt")
print(f"Saving finalized tuned model to {final_save_path}...")
model.save_model(final_save_path)
print("Model firmly saved to disk. Ready for validation!")

Saving finalized tuned model to Z:\ember2024_train_data\ember_model_tuned_full.txt...
Model firmly saved to disk. Ready for validation!
